# Bài 14 · Token & embedding: biến câu chữ thành dãy vectơ

[![Mở trong Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/trungnguyenvn/hoc-sau-notebooks/blob/main/14-token-embedding.ipynb)

Notebook đi kèm [Bài 14 của Học Sâu · Reels](https://d3qr1cjln6qqtk.cloudfront.net/bai-hoc/token-embedding/) (reel và demo tương tác nằm ở trang bài học).
Mạng nơ-ron chỉ nhận số. Ta tự viết bằng Python thuần và numpy, không dùng thư viện tách token nào, cả hai bước biến
văn bản thành số của GPT-2: **bộ tách token BPE cấp byte** (chữ → số hiệu) và **bảng embedding** (số hiệu → vectơ),
trên câu dịch song song thật (FLORES-200) và trên chính các hàng của bảng embedding đã huấn luyện của GPT-2.

**Bạn sẽ làm:**
1. Đổi chữ thành byte UTF-8 và cắt câu thành các “khúc” bằng bộ tách sơ bộ của GPT-2.
2. Tự học một bộ BPE từ đầu: đếm cặp liền kề, gộp cặp hay gặp nhất, lặp lại.
3. Viết lại bộ mã hóa của GPT-2 với đúng 50.000 phép gộp của nó, và đo vì sao tiếng Việt tốn token gấp 4,52 lần.
4. Tra hàng của bảng embedding, đo độ tương đồng cosin và tìm láng giềng gần nhất.
5. Thử phép loại suy $b - a + c$ (vua − đàn ông + phụ nữ), có và không loại các từ đầu vào.

**Cần biết trước:** [Bài 03 · Mạng nhiều lớp & bài toán XOR](https://d3qr1cjln6qqtk.cloudfront.net/bai-hoc/mlp-va-xor/)
(lớp $z = xW + b$) và [Bài 06 · Lan truyền ngược](https://d3qr1cjln6qqtk.cloudfront.net/bai-hoc/lan-truyen-nguoc/).
**Thời gian chạy:** dưới 1 phút trên CPU, kể cả tải FLORES-200 (~26 MB) lần đầu.

Mọi con số trong notebook đều do chính các ô bên dưới tính ra (hoặc đọc từ tệp dữ liệu mà demo dùng), và trùng với
số trong demo.

> Các ô **🤔 Dự đoán trước** đặt một câu hỏi trước khi chạy. Hãy tự trả lời rồi mới chạy ô kế tiếp.

## 1 · Chuẩn bị

Chỉ cần numpy và matplotlib (Colab đã cài sẵn); phần tách token là Python thuần. Hàm `vn()` in số theo kiểu Việt Nam:
dấu phẩy thập phân, dấu chấm hàng nghìn. Python vẫn *gõ* số thập phân bằng dấu chấm (`0.1`), chỉ phần in ra là đổi.

In [ ]:
import base64
import hashlib
import itertools
import json
import math
import re
import sys
import tarfile
import unicodedata
import urllib.request
from collections import Counter
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from matplotlib.colors import LinearSegmentedColormap
from matplotlib.ticker import FixedLocator, FuncFormatter, NullFormatter, NullLocator, ScalarFormatter

# Bảng màu của khóa học (an toàn cho người mù màu); luôn đi kèm một mã hóa thứ hai (hình dạng, nét gạch, nhãn)
BLUE, ORANGE, GREEN, GREY = "#3987e5", "#d95926", "#199e70", "#8a94a6"
DIVERGING = LinearSegmentedColormap.from_list("cam_xam_xanh", [ORANGE, "#f2f2f2", BLUE])  # âm → 0 → dương
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})


def vn(x, digits=2):
    """Số kiểu Việt Nam: 13.007 · 0,1 · −4,38 (không in “−0”)."""
    s = f"{round(float(x), digits) + 0.0:,.{digits}f}".replace(",", "_").replace(".", ",").replace("_", ".")
    return s.replace("-", "−")


def pct(p, digits=2):
    return vn(100 * p, digits) + "%"


def vn_tick(v, _pos=None):
    """Nhãn vạch trên trục: 0,5 · −2 · 60.000 (bỏ số 0 thừa ở cuối)."""
    if abs(v) < 1e-12:
        return "0"
    s = f"{v:.4f}".rstrip("0").rstrip(".")
    return vn(v, len(s.split(".")[1]) if "." in s else 0)


SUP = str.maketrans("-0123456789", "⁻⁰¹²³⁴⁵⁶⁷⁸⁹")


def log_tick(v, _pos=None):
    """Nhãn vạch trục log: 0,5 · 20 · 10⁻¹²."""
    if 0.01 <= v <= 1000:
        return vn_tick(v)
    e = math.floor(math.log10(v) + 1e-9)
    m = v / 10.0**e
    return (f"{vn_tick(m)}·" if abs(m - 1) > 1e-9 else "") + f"10{str(e).translate(SUP)}"


def vn_axes(fig):
    """Đổi nhãn mọi trục của hình (cả thanh màu) sang kiểu Việt Nam; gọi ngay trước plt.show().
    Trục log lấy vạch 1–2–5 nếu ngắn, hoặc các lũy thừa của 10 (cách đều, tối đa ~9 vạch) nếu dài."""
    for ax in fig.axes:
        for axis, lim in ((ax.xaxis, ax.get_xlim()), (ax.yaxis, ax.get_ylim())):
            if axis.get_scale() != "log":
                # only axes still on matplotlib's default number formatter: text labels and custom formatters stay
                if isinstance(axis.get_major_formatter(), ScalarFormatter):
                    axis.set_major_formatter(FuncFormatter(vn_tick))
                continue
            lo, hi = sorted(lim)
            k0, k1 = math.ceil(math.log10(lo) - 1e-9), math.floor(math.log10(hi) + 1e-9)
            if hi / lo < 300:
                ticks = [m * 10.0**k for k in range(k0 - 1, k1 + 1) for m in (1, 2, 5) if lo <= m * 10.0**k <= hi]
            else:
                step = max(1, math.ceil((k1 - k0) / 8))
                ticks = [10.0**k for k in range(k0, k1 + 1) if k % step == 0]
                if step > 1:
                    axis.set_minor_locator(NullLocator())
            axis.set_major_locator(FixedLocator(ticks))
            axis.set_minor_formatter(NullFormatter())
            axis.set_major_formatter(FuncFormatter(log_tick))


print(vn(50257, 0), vn(4.5213, 2), pct(0.480412))

## 2 · Dữ liệu

Ba nguồn, đều kiểm tra mã MD5 nên ai chạy cũng có đúng cùng dữ liệu:
- **FLORES-200** (NLLB Team, 2022; giấy phép CC BY-SA 4.0): cùng một bộ câu tiếng Anh được dịch chuyên nghiệp sang 200
  thứ tiếng. Ta dùng bản tiếng Anh và tiếng Việt: phần *dev* (997 câu) để tự học BPE, phần *devtest* (1.012 câu) để
  đo. Ô dưới tải bản gốc (~26 MB) từ máy chủ của Meta.
- **`gpt2-bpe.json`**: bảng 50.000 phép gộp của GPT-2, dạng cặp số hiệu, cùng các câu mẫu của demo và số token mà
  ba bộ tách token đếm được trên FLORES-200.
- **`gpt2-embed.json`**: 137 hàng thật của bảng embedding GPT-2 bản nhỏ (các từ tiếng Anh có sẵn một token riêng),
  nén int8, cùng các láng giềng và phép loại suy đã tính trên cả 50.257 token. Cả bảng (154 MB ở float32) quá lớn để
  tải về.

Hai tệp JSON là đúng các tệp mà demo trên trang bài học dùng.

In [ ]:
FLORES = ("https://dl.fbaipublicfiles.com/nllb/flores200_dataset.tar.gz", "ceca3bc3a69207c4ae8427266a0b6514")
DATA = Path("data")


def fetch_flores():
    DATA.mkdir(exist_ok=True)
    p = DATA / "flores200_dataset.tar.gz"
    if not p.exists():
        urllib.request.urlretrieve(FLORES[0], p)
    assert hashlib.md5(p.read_bytes()).hexdigest() == FLORES[1], f"{p.name} bị hỏng, hãy xóa và tải lại"
    return p


# Tệp dữ liệu của demo trên web. Khi xuất bản, build.py thay chuỗi "COURSE_DATA_URL" bằng thư mục data/ của repo
# notebook công khai.
def fetch_json(name, md5):
    p = DATA / name
    if not p.exists():
        p.parent.mkdir(exist_ok=True)
        urllib.request.urlretrieve("https://raw.githubusercontent.com/trungnguyenvn/hoc-sau-notebooks/main/data/" + name, p)
    assert hashlib.md5(p.read_bytes()).hexdigest() == md5, f"{name} bị hỏng, hãy xóa và tải lại"
    return json.loads(p.read_text())


flores_path = fetch_flores()
bpe_json = fetch_json("gpt2-bpe.json", "2cd95a0c22ae42cdeda5d6541f4f6f04")  # bảng gộp + số đếm trên FLORES
emb = fetch_json("gpt2-embed.json", "a55436ca8087ea20ef8fb8bdcaa99b15")  # 137 hàng của bảng embedding E
print("đã tải:", flores_path.name, "· gpt2-bpe.json · gpt2-embed.json")

In [ ]:
with tarfile.open(flores_path) as tar:

    def read(split, lang):
        f = tar.extractfile(f"./flores200_dataset/{split}/{lang}.{split}")
        return f.read().decode("utf-8").rstrip("\n").split("\n")

    dev = {"en": read("dev", "eng_Latn"), "vi": read("dev", "vie_Latn")}
    test = {"en": read("devtest", "eng_Latn"), "vi": read("devtest", "vie_Latn")}

print(
    f"dev: {vn(len(dev['vi']), 0)} câu · devtest: {vn(len(test['vi']), 0)} câu (mỗi phần có cả tiếng Anh và tiếng Việt)"
)
for lang in ("en", "vi"):
    print(f"  devtest[0], {lang}: {test[lang][0]}")
assert len(dev["en"]) == len(dev["vi"]) == 997 and len(test["en"]) == len(test["vi"]) == bpe_json["flores"]["n"] == 1012

## 3 · Chữ → byte: UTF-8

Máy tính lưu chữ bằng **điểm mã** Unicode (ô là U+00F4), và lưu điểm mã bằng **byte** theo UTF-8: chữ cái tiếng Anh
tốn 1 byte, phần lớn chữ có dấu của tiếng Việt tốn 2 hoặc 3. Một byte có 256 giá trị, nên “bảng chữ cái” nhỏ nhất
đọc được *mọi* văn bản chỉ có 256 ký hiệu. GPT-2 bắt đầu từ đó: 256 byte là 256 token đầu tiên của nó.

Ô dưới in số byte của vài ký tự, của một câu, và của toàn bộ phần devtest ở hai thứ tiếng.

In [ ]:
for ch in ["a", "ô", "đ", "ệ", "😀"]:
    b = ch.encode("utf-8")
    print(f"{ch!r}: điểm mã U+{ord(ch):04X} → {len(b)} byte: {b.hex(' ').upper()}")

s = "Tôi yêu Việt Nam."
print(f"\n{s!r}: {len(s)} ký tự → {len(s.encode('utf-8'))} byte")
n_char = {lang: sum(len(x) for x in test[lang]) for lang in test}
n_byte = {lang: sum(len(x.encode("utf-8")) for x in test[lang]) for lang in test}
for lang in ("en", "vi"):
    print(f"devtest {lang}: {vn(n_char[lang], 0)} ký tự → {vn(n_byte[lang], 0)} byte")
print(
    f"bản tiếng Việt nhiều hơn {pct(n_char['vi'] / n_char['en'] - 1)} ký tự, "
    f"nhưng nhiều hơn {pct(n_byte['vi'] / n_byte['en'] - 1)} byte"
)

Cùng nội dung, bản tiếng Việt chỉ nhiều hơn bản tiếng Anh 5,02% ký tự nhưng nhiều hơn 38,99% byte, vì mỗi chữ có dấu
tốn 2–3 byte. Các chữ 3 byte (ạ, ả, ấ… ỹ) đều bắt đầu bằng `E1 BA` hoặc `E1 BB`; hãy nhớ hai byte này, ta sẽ gặp lại ở
mục 7.

Một chữ trên màn hình còn có thể có **hai chuỗi byte**. Unicode cho phép viết “ệ” *dựng sẵn* (NFC: một điểm mã) hoặc
*tổ hợp* (NFD: chữ e, rồi dấu nặng, rồi dấu mũ, mỗi dấu là một điểm mã riêng). Văn bản tiếng Việt trên mạng có cả hai.

In [ ]:
for form in ("NFC", "NFD"):
    t = unicodedata.normalize(form, "ệ")
    print(f"{form}: {len(t)} điểm mã {[f'U+{ord(c):04X}' for c in t]} → {len(t.encode('utf-8'))} byte")

## 4 · Cắt câu thành khúc: bộ tách sơ bộ của GPT-2

Trước khi gộp byte, GPT-2 cắt câu thành các **khúc** bằng một biểu thức chính quy, và không phép gộp nào vượt qua
ranh giới khúc. Bài báo GPT-2 giải thích: không có bước này, BPE học ra nhiều bản của cùng một từ (`dog.`, `dog!`,
`dog?`), phí chỗ trong từ điển. Biểu thức gốc của GPT-2 là

```
's|'t|'re|'ve|'m|'ll|'d| ?\p{L}+| ?\p{N}+| ?[^\s\p{L}\p{N}]+|\s+(?!\S)|\s+
```

nghĩa là: các đuôi viết tắt tiếng Anh; một **dãy chữ cái** (kèm tối đa một dấu cách phía trước); một dãy chữ số;
một dãy ký hiệu khác; và khoảng trắng. `\p{L}` là “mọi chữ cái Unicode” và `\p{N}` là “mọi ký tự số”.

Mô-đun `re` của Python không hiểu `\p{L}`. Ta không xấp xỉ mà dựng đúng hai lớp ký tự đó từ bảng Unicode của Python
(`unicodedata.category`: nhóm bắt đầu bằng L là chữ cái, bằng N là số), và viết `\s` thành đúng các ký tự khoảng trắng
của Unicode. (`\s` của Python còn nhận 4 ký tự điều khiển `\x1c`–`\x1f` mà Unicode không coi là khoảng trắng.)

In [ ]:
def char_class(pred):
    """Lớp ký tự cho re: mọi điểm mã thỏa pred, gom thành các khoảng a-b."""
    parts, start = [], None
    for cp in range(sys.maxunicode + 2):
        ok = cp <= sys.maxunicode and pred(chr(cp))
        if ok and start is None:
            start = cp
        elif not ok and start is not None:
            a, b = re.escape(chr(start)), re.escape(chr(cp - 1))
            parts.append(a if cp - 1 == start else f"{a}-{b}")
            start = None
    return "".join(parts), len(parts)


L, n_L = char_class(lambda c: unicodedata.category(c).startswith("L"))  # \p{L}: chữ cái
N, n_N = char_class(lambda c: unicodedata.category(c).startswith("N"))  # \p{N}: số
WS_CHARS = [*range(9, 14), 32, 0x85, 0xA0, 0x1680, *range(0x2000, 0x200B), 0x2028, 0x2029, 0x202F, 0x205F, 0x3000]
WS = re.escape("".join(map(chr, WS_CHARS)))  # 25 ký tự khoảng trắng của Unicode (thuộc tính White_Space)
PAT = re.compile(rf"'s|'t|'re|'ve|'m|'ll|'d| ?[{L}]+| ?[{N}]+| ?[^{WS}{L}{N}]+|[{WS}]+(?![^{WS}])|[{WS}]+")
print(f"Unicode {unicodedata.unidata_version}: lớp chữ cái gồm {n_L} đoạn liên tiếp của bảng mã, lớp số {n_N} đoạn")

for s in ["Tôi yêu Việt Nam.", "GPT-2 có 50.257 token!", "it's I'll HOW'S", "  a  b\n\nc"]:
    print(f"{s!r:28} → {PAT.findall(s)}")

Mỗi khúc là một từ *kèm dấu cách phía trước* (`' yêu'`), một dãy số, hoặc một dãy dấu câu. Vài chi tiết đáng chú ý:
- “50.257” bị cắt thành ba khúc `' 50'`, `'.'`, `'257'`: dấu chấm hàng nghìn của tiếng Việt là dấu câu với GPT-2.
- Đuôi `'s` chỉ được nhận khi viết thường: `HOW'S` thành `' HOW'`, `"'"`, `'S'`.
- Trong `'  a  b'`, dấu cách thứ nhất đứng riêng, dấu cách thứ hai đi với chữ: `\s+(?!\S)` chừa lại dấu cách cuối cùng
  cho từ phía sau.

Một cách xấp xỉ hay gặp là thay `\p{L}` bằng `[^\W\d_]` của Python. Nó gần đúng, nhưng `\w` của Python coi cả các
ký tự số như ² là chữ, `\s` nhận thêm `\x1c`, và dấu gạch dưới `_` thuộc `\w` nên không lớp nào của biểu thức xấp xỉ
nhận nó. Ô dưới so hai cách cắt:

In [ ]:
NAIVE = re.compile(r"'s|'t|'re|'ve|'m|'ll|'d| ?[^\W\d_]+| ?\d+| ?[^\s\w]+|\s+(?!\S)|\s+")
for s in ["x²+y²", "a\x1c!", "my_var", "Tôi yêu Việt Nam."]:
    a, b = PAT.findall(s), NAIVE.findall(s)
    print(f"{s!r:22} đúng: {a}\n{'':22} xấp xỉ: {b}  {'(giống)' if a == b else '(KHÁC)'}")

Hai cách cắt giống nhau trên câu tiếng Việt, nhưng biểu thức xấp xỉ **làm mất** dấu `_` của `my_var` (nó không khớp
lớp nào nên bị bỏ qua), và cắt `x²` khác đi. Với văn bản có code, sai khác này đổi hẳn chuỗi token. Từ đây ta
dùng `PAT`, bản dựng đúng; mục 6 so nó với tiktoken, và mục 7 so tổng số token trên FLORES-200.

## 5 · Tự học BPE từ đầu

**BPE** (Byte Pair Encoding; Sennrich và cộng sự, 2015, mượn từ thuật toán nén của Gage, 1994) học từ điển như sau:
1. Bắt đầu với 256 token, mỗi byte một token. Mỗi khúc là một dãy số hiệu byte.
2. Đếm mọi **cặp liền kề** trong mọi khúc (nhân với số lần khúc đó xuất hiện).
3. Cặp nhiều nhất trở thành token mới, số hiệu 256, 257, …; thay cặp đó bằng token mới ở mọi nơi.
4. Lặp lại bước 2–3 đến khi đủ số phép gộp muốn có.

Kết quả là **danh sách phép gộp có thứ tự**. Ở đây, số hiệu của mỗi byte chính là giá trị của nó (0–255); GPT-2 xếp
256 byte theo một thứ tự khác (mục 6). Khi nhiều cặp cùng nhiều nhất, ta lấy cặp gặp trước.

In [ ]:
def get_stats(words, counts):
    """Số lần xuất hiện của mỗi cặp liền kề; mỗi khúc khác nhau chỉ đếm một lần, nhân với số lần nó xuất hiện."""
    stats = Counter()
    for ids, n in zip(words, counts):
        for pair in itertools.pairwise(ids):
            stats[pair] += n
    return stats


def merge(ids, pair, new_id):
    """Thay mọi lần xuất hiện của cặp `pair` (từ trái sang phải) bằng `new_id`."""
    out, i = [], 0
    while i < len(ids):
        if i < len(ids) - 1 and ids[i] == pair[0] and ids[i + 1] == pair[1]:
            out.append(new_id)
            i += 2
        else:
            out.append(ids[i])
            i += 1
    return out


def train_bpe(chunks, n_merges, log=None):
    """Học n_merges phép gộp từ danh sách khúc. Trả về các cặp đã gộp (theo thứ tự) và bảng số hiệu → byte."""
    chunk_counts = Counter(chunks)
    words = [list(c.encode("utf-8")) for c in chunk_counts]
    counts = list(chunk_counts.values())
    merges, vocab = [], {i: bytes([i]) for i in range(256)}
    total = sum(len(w) * n for w, n in zip(words, counts))  # số token của cả văn bản
    for k in range(n_merges):
        stats = get_stats(words, counts)
        if not stats:
            break
        pair = max(stats, key=stats.get)  # hòa thì lấy cặp gặp trước
        new_id = 256 + k
        merges.append(pair)
        vocab[new_id] = vocab[pair[0]] + vocab[pair[1]]
        merged = [merge(w, pair, new_id) for w in words]
        total -= sum((len(a) - len(b)) * n for a, b, n in zip(words, merged, counts))
        words = merged
        if log is not None:
            log.append((pair, stats[pair], total))
        keep = [i for i, w in enumerate(words) if len(w) > 1]  # khúc đã thành 1 token thì không còn cặp nào
        words, counts = [words[i] for i in keep], [counts[i] for i in keep]
    return merges, vocab


def face(b):
    """Một token để in: chữ nếu các byte của nó là UTF-8 hợp lệ, nếu không thì mã byte, ví dụ <E1 BB>."""
    try:
        return repr(b.decode("utf-8"))
    except UnicodeDecodeError:
        return "<" + b.hex(" ").upper() + ">"

Thử trên 4 câu đầu của *Truyện Kiều* (Nguyễn Du):

In [ ]:
KIEU = (
    "Trăm năm trong cõi người ta,\n"
    "Chữ tài chữ mệnh khéo là ghét nhau.\n"
    "Trải qua một cuộc bể dâu,\n"
    "Những điều trông thấy mà đau đớn lòng."
)
kieu_chunks = PAT.findall(KIEU)
n_bytes = len(KIEU.encode("utf-8"))
print(f"{len(KIEU)} ký tự, {n_bytes} byte, {len(kieu_chunks)} khúc; mở đầu: {kieu_chunks[:8]}")

### 🤔 Dự đoán trước
Phép gộp **đầu tiên** (cặp liền kề hay gặp nhất trong 4 câu này) là gì: một vần như `ng`, một âm đầu như `th`, hay
thứ gì khác?

In [ ]:
log = []
kieu_merges, kieu_vocab = train_bpe(kieu_chunks, 12, log)
print(f"bắt đầu: {n_bytes} token (mỗi byte một token)")
for k, (pair, count, total) in enumerate(log):
    a, b = (kieu_vocab[t] for t in pair)
    print(
        f"gộp {k + 1:>2} → token {256 + k}: {face(a):>12} + {face(b):<6} = {face(a + b):<12} ({count} lần) → {total} token"
    )
assert kieu_vocab[256] == "ữ".encode()[:2] == "ệ".encode()[:2]
assert all(prev - total == count for (_, count, total), prev in zip(log, [n_bytes] + [t for *_, t in log]))

Phép gộp đầu tiên là hai byte `E1 BB`: chưa phải một chữ, chỉ là phần mở đầu chung của các chữ 3 byte ữ, ệ, ộ, ể, ề,
ờ, ớ. BPE không biết chữ cái hay âm tiết là gì; nó chỉ đếm byte. Các token có nghĩa (`ng`, `' t'`, `hữ`) xuất hiện dần
vì chúng là những dãy byte hay lặp lại. Mỗi phép gộp làm văn bản ngắn đi đúng bằng số lần cặp đó xuất hiện (khi các
lần xuất hiện không chồng lên nhau, như ở đây): BPE là một thuật toán nén.

Trên 4 câu, chỉ vài cặp xuất hiện quá 2 lần, nên nhiều cặp hòa nhau. Cần nhiều chữ hơn: ta học 1.000 phép gộp trên 997
câu tiếng Việt của phần dev FLORES-200 (mất vài giây). Phần devtest để dành cho việc đo ở mục 7.

In [ ]:
vi_merges, vi_vocab = train_bpe([c for s in dev["vi"] for c in PAT.findall(s)], 1000)
print("20 phép gộp đầu:", " ".join(face(vi_vocab[256 + k]) for k in range(20)))
print("10 phép gộp cuối:", " ".join(face(vi_vocab[256 + k]) for k in range(990, 1000)))
partial = [t for t in range(256, 1256) if face(vi_vocab[t]).startswith("<")]
print(f"{1000 - len(partial)} token mới là chữ trọn vẹn; {len(partial)} vẫn là mảnh byte, ví dụ", end=" ")
print(*(face(vi_vocab[t]) for t in partial[:4]))

Phần lớn token mới là chữ trọn vẹn (âm tiết, vần, âm đầu…), nhưng vẫn còn những mảnh byte như `E1 BB`: một mảnh hay
gặp được gộp trước, rồi mới thành phần của các token dài hơn. Ta sẽ dùng bảng gộp tự học này ở mục 7 để so với GPT-2.

## 6 · Bộ mã hóa thật của GPT-2

GPT-2 học 50.000 phép gộp theo cùng ý tưởng (OpenAI không công bố mã học), trên dữ liệu gần như toàn tiếng Anh.
Tệp `gpt2-bpe.json` lưu chúng thành các cặp số hiệu: phép gộp thứ $i$ (đếm từ 0) nối hai token có sẵn thành token
$256 + i$. Hai khác biệt nhỏ với bản tự học của ta:
- 256 byte được đánh số theo thứ tự `bytes_to_unicode` của GPT-2: các byte in được (`!`…`~`, `¡`…`¬`, `®`…`ÿ`)
  trước, rồi đến các byte còn lại. Vì vậy dấu cách (byte 32) là token 220, xuống dòng (byte 10) là token 198.
- Tệp `vocab.json` của GPT-2 viết mỗi byte bằng một ký tự in được, nên dấu cách hiện thành `Ġ`: token `' the'` được
  viết là `Ġthe`. Đó chỉ là cách viết trong tệp, không có ký tự `Ġ` nào trong văn bản.

In [ ]:
pairs = np.frombuffer(base64.b64decode(bpe_json["merges"]), dtype="<u2").reshape(-1, 2).tolist()
printable = [*range(ord("!"), ord("~") + 1), *range(ord("¡"), ord("¬") + 1), *range(ord("®"), ord("ÿ") + 1)]
BYTE_ORDER = printable + [b for b in range(256) if b not in printable]  # token i < 256 là byte BYTE_ORDER[i]
BYTE_ID = {b: i for i, b in enumerate(BYTE_ORDER)}
RANK = {tuple(p): i for i, p in enumerate(pairs)}  # cặp → thứ tự của phép gộp

TOKEN_BYTES = [bytes([b]) for b in BYTE_ORDER]
for a, b in pairs:  # mỗi phép gộp chỉ dùng những token có trước nó
    TOKEN_BYTES.append(TOKEN_BYTES[a] + TOKEN_BYTES[b])

n_vocab = 256 + len(pairs) + 1  # + <|endoftext|>, token đặc biệt số 50256
print(f"256 byte + {vn(len(pairs), 0)} phép gộp + 1 token đặc biệt = {vn(n_vocab, 0)} token")
print("token 220 =", TOKEN_BYTES[220], "· token 198 =", TOKEN_BYTES[198], "· token 50255 =", TOKEN_BYTES[50255])
print("10 phép gộp đầu tiên của GPT-2:", " ".join(face(TOKEN_BYTES[256 + i]) for i in range(10)))
print("' the' là token", TOKEN_BYTES.index(b" the"))
assert len(pairs) == 50000 and all(a < 256 + i and b < 256 + i for i, (a, b) in enumerate(pairs))
assert n_vocab == bpe_json["nVocab"] == 50257 and (BYTE_ID[32], BYTE_ID[10]) == (220, 198)

Phép gộp đầu tiên của GPT-2 là `' t'`, rồi `' a'`, `'he'`, `'in'`: dấu vết của tiếng Anh, nơi “ the” là từ hay gặp
nhất (nó là token 262, chỉ sau 7 phép gộp).

**Mã hóa** một khúc mới: bắt đầu từ các byte, rồi lặp lại: trong mọi cặp liền kề, chọn cặp có **thứ tự gộp nhỏ nhất**
trong bảng, gộp nó ở mọi nơi; dừng khi không còn cặp nào có trong bảng. Cách này cho đúng kết quả của việc áp dụng
lần lượt 50.000 phép gộp theo thứ tự đã học, nhưng chỉ tốn vài bước cho mỗi khúc. Đây cũng là cách
[encoder.py](https://github.com/openai/gpt-2/blob/master/src/encoder.py) của OpenAI làm.

In [ ]:
def encode_chunk(chunk, rank=RANK, byte_id=BYTE_ID, k=math.inf, trace=None):
    """Mã hóa một khúc bằng k phép gộp đầu tiên của bảng `rank` (mặc định: cả bảng của GPT-2)."""
    ids = [byte_id[b] for b in chunk.encode("utf-8")]
    while len(ids) > 1:
        pair = min(itertools.pairwise(ids), key=lambda p: rank.get(p, math.inf))
        r = rank.get(pair, math.inf)
        if r >= k:
            break  # không cặp liền kề nào có trong (k phép gộp đầu của) bảng
        ids = merge(ids, pair, 256 + r)
        if trace is not None:
            trace.append((r, ids))
    return ids


CACHE = {}  # mỗi khúc chỉ mã hóa một lần


def encode(text):
    out = []
    for c in PAT.findall(text):
        if c not in CACHE:
            CACHE[c] = encode_chunk(c)
        out += CACHE[c]
    return out


def decode(ids):
    return b"".join(TOKEN_BYTES[t] for t in ids).decode("utf-8", errors="replace")


def show(ids):
    return " | ".join(face(TOKEN_BYTES[t]) for t in ids)


for chunk in [" learning", " Việt"]:
    trace = []
    start = encode_chunk(chunk, k=0)
    encode_chunk(chunk, trace=trace)
    print(f"{chunk!r}: bắt đầu {len(start)} byte: {show(start)}")
    for r, ids in trace:
        print(f"   phép gộp thứ {vn(r, 0):>6} → {show(ids)}   {ids}")

`' learning'` cần 8 phép gộp, mỗi bước một cặp có thứ tự nhỏ nhất. Ở `' Việt'`, chỉ phần `' Vi'` được gộp (2 phép
gộp), còn ba byte của “ệ” và chữ `t` đứng riêng: không phép gộp nào trong 50.000 phép của GPT-2 nhận các byte của “ệ”.

### Kiểm tra với tiktoken

Thư viện [tiktoken](https://github.com/openai/tiktoken) của OpenAI là bộ tách token chính thức. Các số hiệu dưới đây
được tính bằng `tiktoken.get_encoding("gpt2").encode_ordinary(...)` khi viết notebook này, rồi chép vào đây, nên
notebook không cần cài tiktoken. Tệp của demo còn lưu 5 cặp câu mẫu kèm số hiệu do tiktoken tính.

In [ ]:
nfd = unicodedata.normalize("NFD", "Tôi yêu Việt Nam.")  # dấu viết thành ký tự riêng
TIKTOKEN = {  # chuỗi → số hiệu theo tiktoken (bảng "gpt2")
    "Hello world": [15496, 995],
    "it's I'll HOW'S": [270, 338, 314, 1183, 29630, 6, 50],
    "  a  b\n\nc": [220, 257, 220, 275, 198, 198, 66],
    "x²+y²": [87, 31185, 10, 88, 31185],
    "a\x1c!": [64, 216, 0],
    "my_var": [1820, 62, 7785],
    "GPT-2 có 50.257 token!": [38, 11571, 12, 17, 269, 10205, 2026, 13, 28676, 11241, 0],
    "Tiếng Việt": [40533, 157, 118, 123, 782, 16049, 157, 119, 229, 83],
    nfd: [2514, 136, 224, 72, 9838, 136, 224, 84, 47154, 136, 96, 136, 224, 83, 17871, 13],  # dạng NFD
}
for p in bpe_json["presets"]:
    TIKTOKEN[p["vi"]], TIKTOKEN[p["en"]] = p["viIds"], p["enIds"]

for s, ref in TIKTOKEN.items():
    assert encode(s) == ref, f"khác tiktoken: {s!r}"
    assert decode(encode(s)) == s  # mã hóa rồi giải mã phải ra đúng chuỗi ban đầu
print(f"bộ mã hóa của ta == tiktoken trên cả {len(TIKTOKEN)} chuỗi; ví dụ:")
for s in ["Hello world", "Tôi yêu Việt Nam.", "I love Vietnam."]:
    print(f"  {s!r:22} → {encode(s)}\n  {'':22}   {show(encode(s))}")


def encode_naive(text):  # cùng bảng gộp, nhưng cắt khúc bằng biểu thức xấp xỉ
    return [t for c in NAIVE.findall(text) for t in encode_chunk(c)]


wrong = [s for s, ref in TIKTOKEN.items() if encode_naive(s) != ref]
print(f"\nbiểu thức xấp xỉ cho số hiệu sai ở {len(wrong)} chuỗi: {wrong}")

Bộ mã hóa của ta cho đúng từng số hiệu của tiktoken trên mọi chuỗi thử, cả câu mẫu của demo và các trường hợp khó
(khoảng trắng kép, `'s`, ký tự điều khiển, dạng NFD). Bản cắt khúc xấp xỉ sai ở `my_var`; với `x²+y²` và `a\x1c!`
nó cắt khác nhưng tình cờ vẫn ra cùng số hiệu, vì không phép gộp nào nối `x` với `²`. Mục 7 chạy bộ mã hóa trên
toàn bộ FLORES-200 và so tổng số token với số mà tiktoken đếm.

## 7 · Tiếng Việt tốn bao nhiêu token?

### 🤔 Dự đoán trước
Bản tiếng Việt của FLORES-200 nhiều hơn bản tiếng Anh 38,99% byte. Với bộ tách token của GPT-2, nó tốn gấp bao nhiêu
lần số token: khoảng 1,4 lần như số byte, hay nhiều hơn hẳn?

In [ ]:
n_tok = {lang: sum(len(encode(s)) for s in test[lang]) for lang in ("en", "vi")}
units = {lang: sum(len(s.split()) for s in test[lang]) for lang in ("en", "vi")}  # từ / âm tiết: cắt theo dấu cách
for lang, name, unit in (("en", "tiếng Anh", "từ"), ("vi", "tiếng Việt", "âm tiết")):
    print(
        f"{name:10}: {vn(n_tok[lang], 0):>7} token cho {vn(units[lang], 0)} {unit}"
        f" → {vn(n_tok[lang] / units[lang], 2)} token mỗi {unit}"
    )
ratio = n_tok["vi"] / n_tok["en"]
print(f"tiếng Việt tốn gấp {vn(ratio, 2)} lần số token, dù chỉ nhiều hơn {pct(n_byte['vi'] / n_byte['en'] - 1)} byte")
print(
    f"khung ngữ cảnh 1.024 token của GPT-2 chứa khoảng {vn(1024 * units['en'] / n_tok['en'], 0)} từ tiếng Anh, "
    f"nhưng chỉ {vn(1024 * units['vi'] / n_tok['vi'], 0)} âm tiết tiếng Việt"
)
gpt2_count = next(e for e in bpe_json["flores"]["encodings"] if e["id"] == "gpt2")  # tiktoken đếm, lưu trong tệp
assert (n_tok["en"], n_tok["vi"]) == (gpt2_count["en"], gpt2_count["vi"]) == (27044, 122278), "khác với tiktoken"
assert units == bpe_json["flores"]["units"]

**4,52 lần**, gấp hơn ba lần chênh lệch về byte, và tổng số token trùng đúng số tiktoken đếm trên 1.012 cặp câu. Mỗi
từ tiếng Anh tốn trung bình 1,23 token, mỗi âm tiết tiếng Việt 3,98 token: gần như mọi từ tiếng Anh thông dụng có sẵn
một token, còn âm tiết tiếng Việt bị cắt vụn. Cùng một khung ngữ cảnh 1.024 token, GPT-2 “nhìn” được 829 từ tiếng Anh
nhưng chỉ 257 âm tiết tiếng Việt; một dịch vụ tính tiền theo token dùng bộ tách này cũng thu tiếng Việt đắt hơn chừng
ấy lần (Petrov và cộng sự, 2023, gọi đây là sự bất công của bộ tách token giữa các ngôn ngữ).

Vì sao? Phép gộp chỉ dành cho những cặp hay gặp trong văn bản mà bộ tách token được học, và văn bản đó gần như toàn
tiếng Anh: bài báo GPT-2 viết rằng nhóm tác giả *cố ý lọc bỏ các trang không phải tiếng Anh* khỏi WebText, và 10 phép
gộp đầu tiên ở mục 6 đều là mảnh của từ tiếng Anh. Byte của tiếng Việt vì thế hầu như không được gộp.
Hai số đo dưới đây cho thấy điều đó: tỉ lệ token chỉ là **mảnh byte** (không phải một chữ trọn vẹn), và số token của
mỗi từ hoặc âm tiết.

In [ ]:
def is_partial(t):
    """Token chỉ là một phần của một chữ: các byte của nó không phải UTF-8 hợp lệ."""
    try:
        TOKEN_BYTES[t].decode("utf-8")
        return False
    except UnicodeDecodeError:
        return True


chunk_counts = {lang: Counter(c for s in test[lang] for c in PAT.findall(s)) for lang in ("en", "vi")}
for lang in ("en", "vi"):
    share = sum(is_partial(t) for s in test[lang] for t in encode(s)) / n_tok[lang]
    print(f"{lang}: {pct(share)} số token là mảnh byte")
    assert abs(share - bpe_json["flores"]["partial"][lang]) < 6e-5

word = re.compile(rf" ?[{L}]+")  # khúc chỉ gồm chữ cái: một từ / âm tiết, kèm dấu cách phía trước
hist = {}
for lang in ("en", "vi"):
    h = [0] * 6  # 1, 2, …, 5, ≥ 6 token
    for c, n in chunk_counts[lang].items():
        if word.fullmatch(c):
            h[min(len(CACHE[c]), 6) - 1] += n
    hist[lang] = np.array(h)
    print(f"{lang}: {vn(sum(h), 0)} từ/âm tiết; số tốn 1…5, ≥ 6 token: {h} → {pct(h[0] / sum(h))} chỉ cần 1 token")
assert {k: v.tolist() for k, v in hist.items()} == bpe_json["flores"]["chunkHist"]

fig, ax = plt.subplots(figsize=(8, 3.4))
x = np.arange(6)
for k, (lang, label, color, hatch) in enumerate(
    [("en", "tiếng Anh (mỗi từ)", BLUE, None), ("vi", "tiếng Việt (mỗi âm tiết)", ORANGE, "//")]
):
    share = hist[lang] / hist[lang].sum()
    ax.bar(x + (k - 0.5) * 0.38, 100 * share, 0.38, color=color, hatch=hatch, edgecolor="white", label=label)
ax.set_xticks(x, ["1", "2", "3", "4", "5", "≥ 6"])
ax.set(xlabel="số token GPT-2 cần cho một từ / âm tiết", ylabel="tỉ lệ (%)")
ax.set_title("FLORES-200 devtest: số token của mỗi từ / âm tiết", fontsize=11)
ax.legend(frameon=False)
vn_axes(fig)
plt.show()

Gần một nửa số token tiếng Việt (48,04%) chỉ là mảnh byte, so với 0,27% ở tiếng Anh. Trên hình, cột xanh (tiếng Anh)
dồn ở 1 token (94,28% số từ); cột cam gạch chéo (tiếng Việt) trải từ 2 đến hơn 6 token, chỉ 6,28% số âm tiết là một
token.

### Cái giá của dấu
Mỗi chữ cái tiếng Việt khi đứng riêng tốn bao nhiêu token? Hình dưới là 12 nguyên âm × 6 thanh, cùng chữ đ: số trong ô
là số token, ô càng đậm càng tốn.

In [ ]:
VOWELS = ["a", "ă", "â", "e", "ê", "i", "o", "ô", "ơ", "u", "ư", "y"]
TONES = {
    "ngang": "",
    "huyền": chr(0x300),
    "sắc": chr(0x301),
    "hỏi": chr(0x309),
    "ngã": chr(0x303),
    "nặng": chr(0x323),
}  # dấu thanh dạng tổ hợp
letters = [[unicodedata.normalize("NFC", v + m) for m in TONES.values()] for v in VOWELS]
cost = np.array([[len(encode_chunk(ch)) for ch in row] for row in letters])
n_utf8 = np.array([[len(ch.encode("utf-8")) for ch in row] for row in letters])
cost_d = len(encode_chunk("đ"))

fig, ax = plt.subplots(figsize=(6.2, 6.4))
grid = np.vstack([cost, [cost_d] + [0] * 5])
ax.imshow(grid, cmap=LinearSegmentedColormap.from_list("cam", ["#ffffff", ORANGE]), vmin=0, vmax=3.6)
for r, row in enumerate([*letters, ["đ"]]):
    for c, ch in enumerate(row):
        ax.text(c, r, f"{ch} {grid[r, c]}", ha="center", va="center", fontsize=11)
ax.set_xticks(range(6), list(TONES), fontsize=9, rotation=30)
ax.set_yticks(range(13), [*VOWELS, "đ"])
ax.tick_params(length=0)
ax.set_title("Số token GPT-2 cần cho mỗi chữ khi đứng riêng", fontsize=11)
for side in ("left", "bottom"):
    ax.spines[side].set_visible(False)
plt.show()

all_cost = [*cost.ravel(), cost_d]
print("số chữ tốn 1 / 2 / 3 token:", [all_cost.count(k) for k in (1, 2, 3)])
print("tốn 1 token:", " ".join(ch for row, k in zip(letters, cost) for ch, c in zip(row, k) if c == 1))
print(f"{(n_utf8 == 3).sum()} chữ dài 3 byte UTF-8; số token của chúng: {sorted(set(cost[n_utf8 == 3].tolist()))}")
assert [all_cost.count(k) for k in (1, 2, 3)] == [17, 11, 45] and (cost[n_utf8 == 3] == 3).all()

17 chữ chỉ tốn 1 token: 6 nguyên âm không dấu (1 byte) và 11 chữ Latin-1 hai byte như á, à, ê, ô, hay gặp trong
tiếng Pháp, Tây Ban Nha, Bồ Đào Nha. 11 chữ tốn 2 (hai byte UTF-8 không được gộp), và **cả 45 chữ dài 3 byte đều tốn
đúng 3 token**: mỗi byte một token.

### Vì sao đúng 3 token?
Ô dưới đếm, trong cả 50.257 token của GPT-2, những token chứa cặp byte mở đầu `E1 BA` hoặc `E1 BB` của các chữ 3 byte.

In [ ]:
lead = [t for t, b in enumerate(TOKEN_BYTES) if b"\xe1\xba" in b or b"\xe1\xbb" in b]
print("token chứa E1 BA hoặc E1 BB:", len(lead))
assert lead == []

tails = {}  # chữ 3 byte (cả chữ hoa) có hai byte cuối là một token
for cp in range(0x1EA0, 0x1EFA):
    tail = chr(cp).encode("utf-8")[1:]
    if tail in TOKEN_BYTES:
        tails[chr(cp)] = TOKEN_BYTES.index(tail)
print("chữ 3 byte có hai byte cuối là một token:", {ch: (t, face(TOKEN_BYTES[t])) for ch, t in tails.items()})
print(f"'Ồ' → {show(encode_chunk('Ồ'))}  ·  'ồ' → {show(encode_chunk('ồ'))}")

spread = Counter()  # trong câu thật: mỗi chữ 3 byte của devtest nằm trên bao nhiêu token
for c, n in chunk_counts["vi"].items():
    ends = list(itertools.accumulate(len(TOKEN_BYTES[t]) for t in CACHE[c]))
    spans = list(zip([0, *ends[:-1]], ends))  # vị trí byte của từng token trong khúc
    p = 0
    for ch in c:
        k = len(ch.encode("utf-8"))
        if 0x1EA0 <= ord(ch) <= 0x1EF9:
            spread[sum(a < p + k and b > p for a, b in spans)] += n
        p += k
print(f"FLORES-200 devtest: {vn(spread.total(), 0)} chữ 3 byte; số token mà mỗi chữ nằm trên: {dict(spread)}")
assert set(spread) == {3}

**Không có token nào**: hai byte mở đầu chung của 90 chữ ạ…ỹ (cả chữ hoa) chưa từng được gộp với nhau, nên mỗi chữ
ấy ít nhất tốn 2 token. Chỉ có một ngoại lệ tốn 2 thay vì 3: chữ hoa **Ồ** (`E1 BB 92`), vì cặp `BB 92` tình cờ là một
token. Cặp ấy là phần cuối của chữ Hán/Nhật 黒 (`E9 BB 92`), không liên quan gì đến tiếng Việt. Trong câu thật của
FLORES-200, mọi chữ 3 byte đều nằm trên 3 token (byte `E1` đôi khi đi chung token với dấu cách phía trước, `' \xe1'`
là token 28053, nhưng chữ vẫn trải trên 3 token).

### Bộ tách token học trên tiếng Việt
Chi phí này không phải do tiếng Việt, mà do *dữ liệu* mà bộ tách token được học. Ta so ba bảng gộp trên cùng phần
devtest: bảng tự học trên tiếng Việt ở mục 5, một bảng 300 phép gộp tự học trên phần dev tiếng Anh (vài giây), và bảng
của GPT-2, mỗi bảng chỉ dùng $k$ phép gộp đầu tiên. Vì BPE học tham lam từng bước, $k$ phép gộp đầu của một lần học
1.000 phép gộp chính là kết quả của một lần học chỉ $k$ phép gộp.

In [ ]:
en_merges, _ = train_bpe([c for s in dev["en"] for c in PAT.findall(s)], 300)
IDENTITY = {b: b for b in range(256)}  # bảng tự học: số hiệu của byte là chính giá trị của nó
TOKENIZERS = {
    "tự học, tiếng Việt": ({p: i for i, p in enumerate(vi_merges)}, IDENTITY, [0, 30, 100, 300, 1000]),
    "tự học, tiếng Anh": ({p: i for i, p in enumerate(en_merges)}, IDENTITY, [0, 30, 100, 300]),
    "GPT-2": (RANK, BYTE_ID, [0, 30, 100, 300, 1000, 3000, 10000, 50000]),
}


def count_tokens(lang, rank, byte_id, k):
    return sum(len(encode_chunk(c, rank, byte_id, k)) * n for c, n in chunk_counts[lang].items())


curves = {}
for name, (rank, byte_id, ks) in TOKENIZERS.items():
    curves[name] = [(k, count_tokens("en", rank, byte_id, k), count_tokens("vi", rank, byte_id, k)) for k in ks]
    for k, en, vi in curves[name]:
        print(f"{name:18} k = {vn(k, 0):>6}: tiếng Anh {vn(en, 0):>7} · tiếng Việt {vn(vi, 0):>7} → ×{vn(vi / en, 2)}")
assert curves["GPT-2"][-1][1:] == (27044, 122278) and curves["GPT-2"][0][1:] == (n_byte["en"], n_byte["vi"])

Hình dưới vẽ cùng các số đó: trục ngang là số phép gộp $k$ (thang log), trục dọc là tỉ số token tiếng Việt / tiếng
Anh. Đường gạch ở 1 là “hai thứ tiếng tốn như nhau”; ở $k = 0$ (chỉ có byte) tỉ số là 1,39, đúng bằng tỉ số byte.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 3.6))
styles = {
    "tự học, tiếng Việt": (GREEN, "o", "-"),
    "tự học, tiếng Anh": (BLUE, "s", "--"),
    "GPT-2": (ORANGE, "^", "-"),
}
for name, pts in curves.items():
    color, marker, ls = styles[name]
    k = [max(p[0], 1) for p in pts]  # k = 0 vẽ tại 1 trên thang log
    ax.plot(k, [p[2] / p[1] for p in pts], color=color, marker=marker, ls=ls, lw=2, label=name)
ax.axhline(1, color=GREY, lw=1, ls=":")
ax.set_xscale("log")
ax.set(xlabel="số phép gộp k (điểm đầu tiên: k = 0)", ylabel="token tiếng Việt / tiếng Anh")
ax.set_title("Tỉ số token Việt/Anh trên FLORES-200 devtest, theo bảng gộp")
ax.legend(frameon=False)
vn_axes(fig)
plt.show()

Bảng tự học trên tiếng Anh đi gần như trùng đường của GPT-2: học trên tiếng Anh thì tiếng Việt càng gộp càng thiệt.
Bảng tự học trên 997 câu tiếng Việt thì ngược lại: chỉ với 1.000 phép gộp, bản tiếng Việt đã ít token hơn bản tiếng
Anh (×0,54). Các bộ tách token mới hơn của OpenAI có từ điển lớn hơn nhiều, và khoảng cách thu hẹp (số do tiktoken
đếm, lưu trong tệp của demo):

In [ ]:
for e in bpe_json["flores"]["encodings"]:
    print(f"{e['label']:20} từ điển {vn(e['vocab'], 0):>7} token: tiếng Việt / tiếng Anh = ×{vn(e['vi'] / e['en'], 2)}")

## 8 · Token → vectơ: bảng embedding

Số hiệu token chỉ là một cái tên: token 5822 không “lớn hơn” token 582. Mạng nơ-ron cần một vectơ cho mỗi token. GPT-2
giữ một **bảng embedding** $E$ có $V = 50.257$ hàng và $d = 768$ cột: token số $i$ lấy hàng thứ $i$.

$$x = \operatorname{onehot}(i)\,E, \qquad E \in \mathbb{R}^{V \times d}$$

Nhân vectơ one-hot (toàn 0, riêng vị trí $i$ bằng 1) với $E$ chính là lấy hàng $i$, nên trong code đây là phép tra
`E[i]`, không phải phép nhân. Đây cũng là một lớp tuyến tính $z = xW$ như Bài 03, chỉ là đầu vào luôn là one-hot.
Các số trong $E$ được học bằng lan truyền ngược như mọi trọng số khác (Bài 06). Ở lớp vào, gradient chỉ chảy về
những hàng có token xuất hiện trong lô; nhưng GPT-2 dùng lại chính $E$ ở lớp ra (mục 11), nên qua softmax mọi hàng
đều nhận gradient ở mỗi bước.

Ta có 137 hàng thật của $E$ (từ tệp của demo): mỗi hàng lưu dạng int8 (số nguyên −127…127) kèm một hệ số tỉ lệ.
Giải nén là nhân hai thứ đó lại.

In [ ]:
words = [w["w"] for w in emb["words"]]
ids = [w["id"] for w in emb["words"]]
q = np.frombuffer(base64.b64decode(emb["rows"]), dtype=np.int8).reshape(len(words), emb["dim"])
E_rows = q.astype(np.float64) * np.array(emb["scales"])[:, None]  # 137 × 768: các hàng E[id] của GPT-2
W = {w: i for i, w in enumerate(words)}  # từ → vị trí trong E_rows

V, d = emb["nVocab"], emb["dim"]
print(f"E có {vn(V, 0)} × {d} = {vn(V * d, 0)} tham số ({vn(V * d * 4 / 1e6, 0)} MB ở float32)")
print(f"ta có {len(words)} hàng, mỗi hàng {d} số")
print(f"GPT-2 bản nhỏ có 124.439.808 tham số → E chiếm {pct(V * d / 124_439_808, 1)}")
assert V * d == emb["params"] == 38_597_376 and q.shape == (137, 768)

# id của mỗi từ đúng là token của nó (kèm dấu cách phía trước) theo bộ mã hóa ở mục 6
assert all(encode(" " + w) == [i] for w, i in zip(words, ids))
i = W["king"]
print(f"' king' → token {encode(' king')} → hàng {ids[i]} của E; 8 số đầu:", " ".join(vn(v, 3) for v in E_rows[i, :8]))
print(f"độ dài ‖E[{ids[i]}]‖ = {vn(np.linalg.norm(E_rows[i]), 2)}")

Hình dưới vẽ ba hàng của $E$, mỗi hàng 768 số xếp thành lưới 12 × 64: xanh là số dương, cam là số âm, đậm hơn là
xa 0 hơn. Không ai viết tay các số này: chúng được học trong lúc GPT-2 tập đoán token tiếp theo.

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(9, 4.6))
vmax = np.abs(E_rows[[W["king"], W["queen"], W["Monday"]]]).max()
for ax, w in zip(axes, ["king", "queen", "Monday"]):
    im = ax.imshow(E_rows[W[w]].reshape(12, 64), cmap=DIVERGING, vmin=-vmax, vmax=vmax)
    ax.set_title(f"' {w}': token {ids[W[w]]}, hàng {ids[W[w]]} của E", fontsize=10)
    ax.axis("off")
fig.colorbar(im, ax=axes, fraction=0.03)
vn_axes(fig)
plt.show()

### Độ tương đồng cosin
Để so hai vectơ, ta đo góc giữa chúng: $\cos(u, v) = \dfrac{u \cdot v}{\lVert u \rVert\, \lVert v \rVert}$, bằng 1 khi
cùng hướng, 0 khi vuông góc, −1 khi ngược hướng. Chia mỗi hàng cho độ dài của nó trước (vectơ đơn vị) thì cosin chỉ
còn là một tích vô hướng, và cả ma trận cosin là một phép nhân ma trận.

### 🤔 Dự đoán trước
“king” và “Monday” chẳng liên quan gì đến nhau. Cosin giữa hai hàng của chúng gần 0, dương rõ rệt, hay âm?

In [ ]:
U = E_rows / np.linalg.norm(E_rows, axis=1, keepdims=True)  # mỗi hàng dài 1
S = U @ U.T  # 137 × 137: S[i, j] = cos(hàng i, hàng j)
for a, b in [("king", "queen"), ("man", "woman"), ("France", "Paris"), ("king", "apple"), ("king", "Monday")]:
    print(f"cos({a!r}, {b!r}) = {vn(S[W[a], W[b]], 3)}")
off = ~np.eye(len(words), dtype=bool)
print(f"\ntrung bình trên mọi cặp trong 137 từ: {vn(S[off].mean(), 3)}")
r = emb["random"]  # đo trên 199.994 cặp token ngẫu nhiên của cả từ điển (tệp của demo)
print(f"trung bình trên {vn(r['pairs'], 0)} cặp token ngẫu nhiên của cả từ điển: {vn(r['mean'], 4)} ± {vn(r['sd'], 4)}")

Dương rõ rệt: 0,241. Mọi hàng của $E$ cùng lệch về một hướng chung, nên cả hai token chọn ngẫu nhiên cũng có cosin
trung bình 0,2681. Hiện tượng này gọi là **dị hướng** (anisotropy; Ethayarajh, 2019). Vì vậy cosin chỉ có nghĩa khi
*so sánh*: 0,657 của king/queen chỉ đáng chú ý vì nó lớn hơn hẳn mức nền 0,27.

Hai cách kiểm tra rằng mức nền này do huấn luyện mà có:
- **Lúc khởi tạo**, GPT-2 rút mỗi số của $E$ từ $\mathcal N(0;\ 0{,}02^2)$. Hai vectơ ngẫu nhiên 768 chiều gần như
  vuông góc: cosin quanh 0 với độ lệch chuẩn cỡ $1/\sqrt{768}$.
- **Trừ vectơ trung bình** của các hàng đi thì hướng chung biến mất, và các cặp không liên quan về quanh 0.

In [ ]:
rng = np.random.default_rng(0)
E_init = rng.normal(0, 0.02, size=E_rows.shape)  # mô phỏng 137 hàng lúc khởi tạo
U_init = E_init / np.linalg.norm(E_init, axis=1, keepdims=True)
S_init = U_init @ U_init.T
print(
    f"lúc khởi tạo: cos trung bình {vn(S_init[off].mean(), 3)} ± {vn(S_init[off].std(), 3)}"
    f" (1/√768 = {vn(1 / math.sqrt(768), 3)})"
)

C_rows = E_rows - E_rows.mean(0)  # trừ vectơ trung bình của 137 hàng
Uc = C_rows / np.linalg.norm(C_rows, axis=1, keepdims=True)
Sc = Uc @ Uc.T
for a, b in [("king", "queen"), ("king", "Monday")]:
    print(f"sau khi trừ trung bình: cos({a!r}, {b!r}) = {vn(Sc[W[a], W[b]], 3)}  (trước: {vn(S[W[a], W[b]], 3)})")
print(f"sau khi trừ trung bình: cos trung bình mọi cặp {vn(Sc[off].mean(), 3)}")
assert abs(S_init[off].mean()) < 0.01 and S[off].mean() > 0.2 and S[W["king"], W["Monday"]] > 0.2

Bản đồ cosin của 16 từ thuộc 4 nhóm cho thấy cả hai điều. Bên trái là cosin gốc; bên phải là sau khi trừ trung bình.
Màu xanh là dương, cam là âm, cùng thang −1…1; nhãn ở hai trục là các từ, kẻ ô theo nhóm.

In [ ]:
group = ["king", "queen", "man", "woman", "France", "Paris", "Japan", "Tokyo"]
group += ["Monday", "Tuesday", "seven", "eight", "go", "went", "walk", "walked"]
gi = [W[w] for w in group]
fig, axes = plt.subplots(1, 2, figsize=(10, 4.9))
for ax, M, title in [(axes[0], S, "cosin gốc"), (axes[1], Sc, "sau khi trừ vectơ trung bình")]:
    im = ax.imshow(M[np.ix_(gi, gi)], cmap=DIVERGING, vmin=-1, vmax=1)
    ax.set_xticks(range(16), group, rotation=90, fontsize=8)
    ax.set_yticks(range(16), group, fontsize=8)
    for k in (3.5, 7.5, 11.5):
        ax.axhline(k, color="black", lw=0.8)
        ax.axvline(k, color="black", lw=0.8)
    ax.set_title(title, fontsize=10)
fig.colorbar(im, ax=axes, fraction=0.025)
vn_axes(fig)
plt.show()

Bên trái hầu như mọi ô đều xanh: mức nền 0,27 của dị hướng. Bên phải, bốn khối trên đường chéo vẫn xanh (gia đình và
giới tính, nước và thủ đô, thứ và số, động từ và dạng quá khứ của nó), còn các cặp khác nhóm chuyển sang trắng hoặc
cam.

### Láng giềng gần nhất
Tệp của demo lưu 10 láng giềng gần nhất của mỗi từ trong **cả 50.257 token** (tính bằng float64 trên bảng đầy đủ).
Ta in danh sách của vài từ, và kiểm tra những láng giềng nằm trong 137 hàng của ta: cosin tính lại từ hàng int8 phải
khớp cosin đã lưu, sai lệch dưới 0,003.

In [ ]:
strings = emb["strings"]  # số hiệu → chữ của token, cho những token mà tệp nhắc đến
pos = {t: k for k, t in enumerate(ids)}  # số hiệu → vị trí trong 137 hàng
err = 0.0
for k, nb in enumerate(emb["neighbours"]):
    for t, c in nb:
        if t in pos:
            err = max(err, abs(S[k, pos[t]] - c))
print(f"sai lệch lớn nhất giữa cosin tính lại (int8) và cosin đã lưu (float64): {vn(err, 4)} < 0,003")
assert err < 0.003

for w in ["king", "Monday", "seven", "Paris"]:
    nb = emb["neighbours"][W[w]][:6]
    print(f"{w!r:9} → " + ", ".join(f"{strings[str(t)]!r} {vn(c, 2)}" for t, c in nb))

Các láng giềng gần nhất cho thấy embedding gom những token hay đứng trong ngữ cảnh giống nhau: thứ trong tuần,
số đếm, thành phố. Danh sách còn có các biến thể viết hoa và không có dấu cách phía trước (`'Paris'`, `'King'`): với
GPT-2 đó là những token khác nhau, mỗi token một hàng riêng, và mô hình phải tự học rằng chúng gần nghĩa.

## 9 · Phép loại suy: $b - a + c$

Mikolov và cộng sự (2013) thấy rằng hiệu của hai vectơ từ có thể mang một quan hệ: vectơ(“King”) − vectơ(“Man”) +
vectơ(“Woman”) nằm gần nhất vectơ(“Queen”). Cách làm: lấy vectơ đơn vị của ba từ, tính $v = \hat b - \hat a + \hat c$
(ở đây $a$ = man, $b$ = woman, $c$ = king), rồi tìm từ có cosin lớn nhất với $v$. Một chi tiết hay bị bỏ qua: bài báo
word2vec viết rằng họ **loại các từ của câu hỏi** khỏi phép tìm kiếm.

### 🤔 Dự đoán trước
Với các hàng thật của $E$ trong GPT-2, nếu *không* loại ba từ đầu vào, từ nào gần $v$ nhất?

In [ ]:
def analogy(a, b, c, exclude=True, top=5):
    """“a với b cũng như c với ?”: xếp hạng 137 từ theo cosin với b̂ − â + ĉ."""
    v = U[W[b]] - U[W[a]] + U[W[c]]
    s = U @ (v / np.linalg.norm(v))
    order = [k for k in np.argsort(-s) if not (exclude and words[k] in (a, b, c))]
    return [(words[k], float(s[k])) for k in order[:top]]


keep, drop = analogy("man", "woman", "king", exclude=False), analogy("man", "woman", "king", exclude=True)
print("giữ từ đầu vào:", ", ".join(f"{w} {vn(c, 3)}" for w, c in keep))
print("loại từ đầu vào:", ", ".join(f"{w} {vn(c, 3)}" for w, c in drop))

relations = {r["id"]: r for r in emb["relations"]}
stored = next(r for r in relations["gender"]["results"] if (r["a"], r["b"], r["c"]) == ("man", "woman", "king"))
for key, name in (("incl", "giữ"), ("excl", "loại")):
    print(f"cả 50.257 token, {name}:", ", ".join(f"{strings[str(t)]!r} {vn(c, 3)}" for t, c in stored[key]))
    in_ours = [t for t, _ in stored[key] if t in pos]  # các token của danh sách đã lưu có trong 137 hàng của ta
    ours = analogy("man", "woman", "king", exclude=key == "excl", top=len(in_ours))
    assert [ids[W[w]] for w, _ in ours] == in_ours  # cùng thứ tự
assert keep[0][0] == "king" and keep[1][0] == "queen" and drop[0][0] == "queen"

Gần nhất là **chính “king”** (0,740); “queen” đứng thứ hai (0,694), và chỉ lên hạng 1 khi ta loại ba từ đầu vào. Danh
sách trên cả từ điển cho cùng kết quả. Lý do nằm ở độ dài: $v = \hat c + (\hat b - \hat a)$ chứa nguyên vectơ đơn vị
của “king”, còn phần dịch chuyển $\hat b - \hat a$ ngắn hơn, vì “man” và “woman” vốn đã gần nhau:

In [ ]:
offset = U[W["woman"]] - U[W["man"]]
print(f"‖woman − man‖ = {vn(np.linalg.norm(offset), 3)}  so với ‖king‖ = 1 (vectơ đơn vị)")
print(
    f"cos(man, woman) = {vn(S[W['man'], W['woman']], 3)} → ‖woman − man‖ = √(2 − 2·cos) = {vn(math.sqrt(2 - 2 * S[W['man'], W['woman']]), 3)}"
)

Vậy câu “king − man + woman = queen” đúng một nửa: hướng “nam → nữ” có thật (nó đẩy “queen” lên trên “prince”,
“kings”), nhưng phép tính chỉ thắng khi ta cấm câu trả lời hiển nhiên nhất. Nissim và cộng sự (2020) chỉ ra mặt trái
của luật này: với câu “man : doctor = woman : ?”, phép tìm kiếm *không được phép* trả lời “doctor”, kể cả khi đó là
đáp án đúng, nên kết quả bị ép (“nurse”…) đã bị đọc như bằng chứng về định kiến.

Tệp của demo chạy mọi cặp câu hỏi trong 6 quan hệ (580 câu) trên cả từ điển. Ta tính lại cùng 580 câu trên 137 hàng
int8 và kiểm tra: mỗi điểm cosin trong danh sách đã lưu, nếu token ấy có trong 137 hàng, phải khớp điểm ta tính lại.

In [ ]:
err, n_q, top_is_input, top_is_c = 0.0, 0, 0, 0
for r in emb["relations"]:
    for res in r["results"]:
        a, b, c = res["a"], res["b"], res["c"]
        v = U[W[b]] - U[W[a]] + U[W[c]]
        s = U @ (v / np.linalg.norm(v))
        for t, score in res["incl"] + res["excl"]:
            if t in pos:
                err = max(err, abs(s[pos[t]] - score))
        top_is_input += words[int(np.argmax(s))] in (a, b, c)
        top_is_c += words[int(np.argmax(s))] == c
        n_q += 1
print(f"{n_q} câu hỏi; sai lệch điểm lớn nhất so với tệp: {vn(err, 4)}")
idof = dict(zip(words, ids))
full = sum(r["incl"][0][0] in {idof[r[k]] for k in "abc"} for rel in emb["relations"] for r in rel["results"])
print("không loại từ đầu vào, hạng 1 là một trong ba từ đầu vào:")
print(f"  trên 137 từ: {top_is_input}/{n_q} câu ({pct(top_is_input / n_q)}), trong đó {top_is_c} câu là chính c")
print(f"  trên cả 50.257 token (tệp): {full}/{n_q} câu ({pct(full / n_q)})")
assert n_q == sum(r["n"] for r in emb["relations"]) == 580 and err < 0.003

Không loại từ đầu vào thì khoảng 7 trên 10 câu trả lời là một trong ba từ của câu hỏi, gần như cùng tỉ lệ trên 137 từ
và trên cả từ điển. Gần như luôn là chính $c$ (ở ví dụ trên là “king”), vì $v$ chứa nguyên $\hat c$.

### Kết quả theo quan hệ, và một mốc so sánh
Bảng dưới đếm tỉ lệ câu có đáp án đúng ở hạng 1, theo ba cách chấm:
- **giữ** và **loại** ba từ đầu vào;
- **chỉ lấy c**: bỏ qua $a$ và $b$, trả lời bằng từ gần $c$ nhất (vẫn loại ba từ đầu vào). Linzen (2016) đề nghị mốc
  này: nếu nó đã đúng nhiều, phần lớn “thành công” của phép loại suy đến từ chỗ đáp án vốn nằm gần $c$.

Mỗi cách chấm được tính hai lần: trên 137 từ của ta (ứng viên ít, tính ngay ở đây), và trên cả 50.257 token (đọc từ
tệp của demo).

In [ ]:
def score(rel, pool_scores):
    """Tỉ lệ đúng ở hạng 1 của một quan hệ: giữ / loại / chỉ lấy c, trên 137 từ."""
    hits = np.zeros(3)
    for res in rel["results"]:
        a, b, c, d = (W[res[k]] for k in "abcd")
        v = U[b] - U[a] + U[c]
        s = U @ (v / np.linalg.norm(v))
        hits[0] += np.argmax(s) == d
        s[[a, b, c]] = -np.inf
        hits[1] += np.argmax(s) == d
        sc = pool_scores[c].copy()
        sc[[a, b, c]] = -np.inf
        hits[2] += np.argmax(sc) == d
    return hits / rel["n"]


print(f"{'quan hệ':24}{'câu':>5} │ {'137 từ: giữ · loại · chỉ c':>28} │ {'50.257 token: giữ · loại · chỉ c':>32}")
for r in emb["relations"]:
    ours = score(r, S)
    theirs = (r["incl"], r["excl"], r["onlyC"])
    print(
        f"{r['label']:24}{r['n']:>5} │ {' · '.join(pct(x, 0) for x in ours):>28} │ {' · '.join(pct(x, 0) for x in theirs):>32}"
    )
g = emb["google"]
print(
    f"\nbộ câu hỏi Google (Mikolov và cộng sự, 2013), {vn(g['n'], 0)}/{vn(g['total'], 0)} câu có đủ 4 từ là token, "
    f"cả từ điển: giữ {pct(g['incl'])} · loại {pct(g['excl'])} · chỉ c {pct(g['onlyC'])}"
)

Ba điều đọc được từ bảng:
- **Loại từ đầu vào** đổi kết quả nhiều nhất: trên bộ câu hỏi Google, từ 11,15% lên 78,76%.
- **Mốc “chỉ lấy c”** đã đúng 32,26% trên bộ Google, và với số nhiều, quá khứ thì đúng 100% trên 137 từ: “cats” vốn
  là từ gần “cat” nhất. Phép loại suy chỉ thật sự thêm được thông tin khi nó vượt xa mốc này (như ở thủ đô).
- **Tập ứng viên** quan trọng: với 137 ứng viên, tỉ lệ đúng cao hơn hẳn so với cả từ điển (thủ đô: 86% so với 62%),
  vì có ít từ gây nhiễu hơn. Hai con số chỉ so được khi chấm trên cùng một tập.

### Hướng “nước → thủ đô”
Nếu quan hệ là một hướng cố định, các mũi tên từ mỗi nước đến thủ đô của nó phải gần song song. Hình dưới chiếu 22 từ
của quan hệ này xuống 2 chiều bằng PCA (hai hướng có phương sai lớn nhất của chúng; Bài 19 so PCA với autoencoder):
chấm tròn xanh là nước, vuông cam là thủ đô, mũi tên xám nối từng cặp.

In [ ]:
pairs_cap = relations["capital"]["pairs"]
cap_words = [w for pair in pairs_cap for w in pair]
X = U[[W[w] for w in cap_words]]
X = X - X.mean(0)
_, sv, Vt = np.linalg.svd(X, full_matrices=False)
P2 = X @ Vt[:2].T  # tọa độ trên hai hướng chính
P2 *= np.sign((P2[1::2] - P2[::2]).mean(0))  # dấu của hướng chính tùy máy: lật để mũi tên trung bình chỉ lên, sang phải
xy = dict(zip(cap_words, P2))
LABEL = {  # (dx, dy, căn lề) theo point, cho các nhãn nằm sát nhau; các từ khác dùng vị trí mặc định
    "Spain": (6, 10, "left"), "France": (10, -1, "left"), "Italy": (6, -8, "left"), "Germany": (6, -4, "left"),
    "Rome": (-6, -2, "right"), "Madrid": (0, 8, "center"), "Paris": (5, -2, "left"),
    "Tokyo": (-6, 0, "right"), "Beijing": (0, 8, "center"), "Seoul": (0, -14, "center"), "Bangkok": (6, 0, "left"),
}  # fmt: skip
fig, ax = plt.subplots(figsize=(7.5, 5.2))
for country, city in pairs_cap:
    ax.annotate("", xy[city], xy[country], arrowprops={"arrowstyle": "->", "color": GREY, "lw": 1.2})
    ax.scatter(*xy[country], marker="o", s=45, color=BLUE, zorder=3)
    ax.scatter(*xy[city], marker="s", s=45, color=ORANGE, zorder=3)
    for w, default in ((country, (4, 4, "left")), (city, (4, -10, "left"))):
        dx, dy, ha = LABEL.get(w, default)
        ax.annotate(w, xy[w], xytext=(dx, dy), textcoords="offset points", fontsize=8, ha=ha, va="center")
ax.set_xlim(right=P2[:, 0].max() + 0.12)  # chừa chỗ cho nhãn bên phải
ax.set(xlabel="hướng chính 1", ylabel="hướng chính 2")
ax.set_title("Nước (tròn) → thủ đô (vuông): 11 cặp, chiếu PCA", fontsize=11)
vn_axes(fig)
plt.show()
print(f"hai hướng chính giữ {pct((sv[:2] ** 2).sum() / (sv**2).sum(), 1)} phương sai của 22 vectơ")

rng = np.random.default_rng(1)
for rid in ("capital", "gender", "plural", "past"):
    D = np.array([U[W[b]] - U[W[a]] for a, b in relations[rid]["pairs"]])  # mỗi hàng: hiệu của một cặp
    D /= np.linalg.norm(D, axis=1, keepdims=True)
    C = D @ D.T
    print(
        f"{relations[rid]['label']:24}: cos trung bình giữa các hiệu b̂ − â = {vn(C[np.triu_indices(len(D), 1)].mean(), 2)}"
    )
R = rng.integers(0, len(words), size=(200, 2))
R = R[R[:, 0] != R[:, 1]]
D = U[R[:, 1]] - U[R[:, 0]]
D /= np.linalg.norm(D, axis=1, keepdims=True)
print(f"{'hiệu của các cặp từ ngẫu nhiên':24}: {vn((D @ D.T)[np.triu_indices(len(D), 1)].mean(), 2)}")

Bảng in tỉ lệ đúng của từng quan hệ, cả trên 137 từ và trên cả từ điển. Hình chiếu cho thấy các mũi tên nước → thủ đô
cùng chỉ về một phía, nhưng không song song hoàn toàn. Các số cosin cũng nói vậy: hiệu $\hat b - \hat a$ của hai cặp
cùng quan hệ có cosin trung bình 0,31–0,48, còn hiệu của hai cặp từ ngẫu nhiên thì quanh 0. Hướng của quan hệ có thật,
nhưng chỉ là một phần của mỗi hiệu.

## 10 · Thử thay đổi

Các ô dưới chạy lại cùng các hàm với đầu vào khác. Bạn có thể sửa giá trị rồi chạy lại.

### Câu của bạn

In [ ]:
my_text = "Học sâu bắt đầu từ những con số."  # sửa câu này
my_ids = encode(my_text)
print(f"{len(my_text)} ký tự, {len(my_text.encode('utf-8'))} byte → {len(my_ids)} token")
print(my_ids)
print(show(my_ids))
assert decode(my_ids) == my_text

### Dạng Unicode và dấu
Cùng 1.012 câu tiếng Việt, viết ở dạng tổ hợp (NFD: dấu là ký tự riêng) hoặc bỏ hết dấu (đ → d):

In [ ]:
def strip_marks(s):
    """Bỏ dấu: tách dấu ra (NFD), xóa các dấu tổ hợp, đổi đ thành d."""
    s = unicodedata.normalize("NFD", s.replace("đ", "d").replace("Đ", "D"))
    return unicodedata.normalize("NFC", "".join(ch for ch in s if not unicodedata.combining(ch)))


print(strip_marks("Tôi yêu Việt Nam. Đà Nẵng"))
forms = {
    "dựng sẵn (NFC)": lambda s: unicodedata.normalize("NFC", s),
    "tổ hợp (NFD)": lambda s: unicodedata.normalize("NFD", s),
    "bỏ hết dấu": strip_marks,
}
for name, f in forms.items():
    n = sum(len(encode(f(s))) for s in test["vi"])
    print(f"{name:15}: {vn(n, 0):>7} token → ×{vn(n / n_tok['en'], 2)} so với tiếng Anh")

FLORES-200 viết ở dạng NFC, nên dòng đầu trùng với mục 7. Ở dạng NFD, mỗi dấu thành một ký tự 2 byte riêng và số token
còn tăng nữa. Bỏ hết dấu thì tiếng Việt chỉ còn tốn khoảng gấp đôi tiếng Anh: hơn nửa chi phí nằm ở các byte của dấu.
Nhưng bỏ dấu làm mất nghĩa (“ma, má, mà, mả, mã, mạ” thành cùng một chữ), nên lời giải đúng là một bảng gộp học trên
tiếng Việt, không phải sửa văn bản.

### Bộ tách token song ngữ
Học 300 phép gộp trên cả 997 câu tiếng Anh và 997 câu tiếng Việt của phần dev (vài giây). Thử đổi `n_mix`:

In [ ]:
n_mix = 300
mix_merges, _ = train_bpe([c for lang in ("en", "vi") for s in dev[lang] for c in PAT.findall(s)], n_mix)
mix_rank = {p: i for i, p in enumerate(mix_merges)}
en_n, vi_n = (count_tokens(lang, mix_rank, IDENTITY, n_mix) for lang in ("en", "vi"))
print(f"song ngữ, {n_mix} phép gộp: tiếng Anh {vn(en_n, 0)} · tiếng Việt {vn(vi_n, 0)} → ×{vn(vi_n / en_n, 2)}")
for name in ("tự học, tiếng Việt", "tự học, tiếng Anh"):
    k, en, vi = next(p for p in curves[name] if p[0] == 300)
    print(f"so với {name}, 300 phép gộp: ×{vn(vi / en, 2)}")

### Phép loại suy khác
Mỗi dòng in 3 từ đứng đầu khi giữ và khi loại từ đầu vào. Mọi từ phải nằm trong 137 từ của tệp
(`print(words)` để xem danh sách).

In [ ]:
for a, b, c in [
    ("France", "Paris", "Japan"),
    ("big", "bigger", "small"),
    ("go", "went", "see"),
    ("man", "king", "woman"),
]:
    keep = ", ".join(w for w, _ in analogy(a, b, c, exclude=False, top=3))
    drop = ", ".join(w for w, _ in analogy(a, b, c, top=3))
    print(f"{b} − {a} + {c}: giữ → {keep:28} loại → {drop}")

## 11 · Giới hạn và bước tiếp theo

Đến đây, một câu đã thành một dãy vectơ: tách token, rồi tra mỗi token một hàng của $E$. Bảng $E$ có hai giới hạn
lớn.

**Embedding là tĩnh.** Hàng của một token giống nhau trong mọi câu, mọi ngữ cảnh. Bốn âm tiết tiếng Việt “hay”, “cho”,
“con”, “ra” tình cờ là token của GPT-2 vì chúng cũng là chữ (hoặc mảnh chữ) tiếng Anh, nên hàng của chúng mang
nghĩa tiếng Anh: “hay” là cỏ khô, “cho” đứng cạnh “helicopter” (chopper).

In [ ]:
for w in ["hay", "cho"]:
    nb = emb["neighbours"][W[w]][:6]
    print(f"{w!r} (âm tiết tiếng Việt, token {ids[W[w]]}) → " + ", ".join(f"{strings[str(t)]!r}" for t, _ in nb))

**Embedding không biết thứ tự.** Hai câu ngược nghĩa nhau có thể dùng đúng cùng một tập token. Nếu mô hình chỉ cộng
(hay lấy trung bình) các vectơ của chúng, hai câu cho cùng một vectơ:

In [ ]:
for pair in [(" chó cắn người", " người cắn chó"), (" cat ate mouse", " mouse ate cat")]:
    t1, t2 = (encode(s) for s in pair)
    print(f"{pair[0]!r}: {t1}\n{pair[1]!r}: {t2}\n  cùng tập token: {sorted(t1) == sorted(t2)}")
    assert sorted(t1) == sorted(t2) and t1 != t2
rows = {s: E_rows[[pos[t] for t in encode(s)]] for s in (" cat ate mouse", " mouse ate cat")}
m1, m2 = (r.mean(0) for r in rows.values())
print(f"trung bình embedding của hai câu tiếng Anh: sai khác lớn nhất {np.abs(m1 - m2).max():.0e} (chỉ do làm tròn)")
assert np.allclose(m1, m2)

Để biết ai cắn ai, mô hình phải đọc các vectơ **theo thứ tự**, và một token phải được hiểu nhờ các token quanh nó.
Bài 15 làm điều đó bằng một mạng hồi quy, đọc từng token và mang theo một trạng thái. Bài 16–17 làm bằng cơ chế chú
ý; GPT-2 còn cộng vào mỗi token một vectơ vị trí $P[t]$ (bảng $P$ có 1.024 × 768 số, Bài 17). Ở đầu ra, GPT-2 dùng lại
chính bảng $E$: điểm của token tiếp theo là tích vô hướng của trạng thái cuối với từng hàng của $E$.

## Tóm tắt
- Văn bản là byte UTF-8: 256 byte đọc được mọi chuỗi, nên GPT-2 không bao giờ cần token “không biết”. Chữ tiếng Việt
  có dấu tốn 2–3 byte; bản tiếng Việt của FLORES-200 nhiều hơn bản tiếng Anh 5,02% ký tự nhưng 38,99% byte.
- Bộ tách sơ bộ cắt câu thành khúc (từ kèm dấu cách, số, dấu câu); phép gộp không vượt qua ranh giới khúc. Bản dựng
  đúng `\p{L}`/`\p{N}` cho đúng số hiệu của tiktoken; bản xấp xỉ bằng `\w` làm mất dấu `_`.
- BPE học bằng cách lặp lại “gộp cặp liền kề nhiều nhất”, và mã hóa bằng cách áp lại các phép gộp theo thứ tự đã học.
  Từ điển GPT-2 = 256 byte + 50.000 phép gộp + 1 token đặc biệt = 50.257.
- Trên FLORES-200, tiếng Việt tốn gấp 4,52 lần số token (3,98 token mỗi âm tiết; 48,04% token là mảnh byte), vì bảng
  gộp học từ văn bản tiếng Anh. Không token nào chứa `E1 BA` hay `E1 BB`, nên 45 chữ 3 byte luôn tốn 3 token. Một bảng
  1.000 phép gộp học trên tiếng Việt đảo ngược tỉ số (×0,54).
- Embedding là tra bảng: token $i$ lấy hàng $i$ của $E$ (50.257 × 768 = 38.597.376 tham số, 31,0% GPT-2 bản nhỏ).
- Cosin chỉ có nghĩa khi so với mức nền: hai token bất kỳ có cos trung bình 0,2681 (dị hướng), còn lúc khởi tạo là 0.
- king − man + woman: gần nhất là chính “king”; “queen” chỉ đứng đầu khi loại ba từ đầu vào (bộ Google: 11,15% →
  78,76%).

## Câu hỏi ôn tập
1. Vì sao bộ tách token cấp byte của GPT-2 không cần token “không biết” (`[UNK]`), dù được học gần như chỉ trên tiếng
   Anh?
2. Phép gộp đầu tiên học trên 4 câu Kiều là `E1 BB`, không phải một chữ cái. Vì sao BPE có thể tạo ra token như vậy?
3. Khi mã hóa, vì sao phải áp các phép gộp theo đúng thứ tự đã học, thay vì chọn cách cắt nào ít token nhất?
4. Bản tiếng Việt chỉ nhiều hơn 38,99% byte nhưng tốn gấp 4,52 lần token. Phần chênh lệch đến từ đâu? Nó ảnh hưởng gì
   đến khung ngữ cảnh 1.024 token của GPT-2?
5. cos(“king”, “Monday”) = 0,241. Có nên kết luận hai từ này “hơi giống nghĩa”? Cần so với gì?
6. Vì sao vectơ gần $\widehat{\text{woman}} - \widehat{\text{man}} + \widehat{\text{king}}$ nhất lại là chính “king”?

<details><summary>Gợi ý đáp án</summary>

1. 256 byte đầu tiên đều là token, và mọi chuỗi Unicode đều là một dãy byte UTF-8. Chữ lạ chỉ tốn nhiều token hơn
   (mỗi byte một token), không bao giờ bị mất.
2. BPE chỉ đếm cặp byte liền kề, không biết ranh giới chữ cái. `E1 BB` là phần mở đầu chung của nhiều chữ 3 byte (ữ,
   ệ, ộ…), nên nó là cặp hay gặp nhất dù chưa phải một chữ.
3. Mô hình được huấn luyện trên đúng cách cắt đó: mỗi token có một hàng của $E$ đã học theo các ngữ cảnh nó xuất hiện.
   Cắt khác đi (dù ít token hơn) sẽ đưa vào những token mà mô hình gần như chưa thấy ở vị trí đó. Áp theo thứ tự cũng
   bảo đảm một chuỗi luôn cho cùng một dãy token.
4. Từ bảng gộp: các phép gộp được học trên văn bản tiếng Anh, nên gần một nửa token tiếng Việt chỉ là mảnh byte
   (48,04% so với 0,27%). Khung 1.024 token chứa khoảng 829 từ tiếng Anh nhưng chỉ 257 âm tiết tiếng Việt.
5. Không: hai token ngẫu nhiên đã có cos trung bình 0,2681 vì mọi hàng của $E$ lệch về một hướng chung. 0,241 nằm
   ngay ở mức nền; sau khi trừ vectơ trung bình, nó còn −0,033. Phải so với mức nền (hoặc trừ trung bình) trước.
6. Vectơ đó chứa nguyên vectơ đơn vị của “king”, còn phần dịch chuyển woman − man ngắn hơn (dài 0,845, vì man và woman
   vốn gần nhau), nên nó vẫn gần “king” nhất. Bài báo word2vec loại các từ của câu hỏi khỏi phép tìm kiếm.
</details>

## Tìm hiểu thêm
- R. Sennrich, B. Haddow, A. Birch (2015), [Neural Machine Translation of Rare Words with Subword Units](https://arxiv.org/abs/1508.07909):
  BPE cho mạng nơ-ron.
- A. Radford và cộng sự (2019), [Language Models are Unsupervised Multitask Learners](https://cdn.openai.com/better-language-models/language_models_are_unsupervised_multitask_learners.pdf):
  mục 2.2 về BPE cấp byte và bộ tách sơ bộ; mã gốc [encoder.py](https://github.com/openai/gpt-2/blob/master/src/encoder.py).
- A. Karpathy, [Let's build the GPT Tokenizer](https://www.youtube.com/watch?v=zduSFxRajkE) (video) và
  [minbpe](https://github.com/karpathy/minbpe): dựng lại cùng thuật toán, có bài tập.
- Hugging Face LLM Course, [Byte-Pair Encoding tokenization](https://huggingface.co/learn/llm-course/chapter6/5).
- OpenAI, [tiktoken](https://github.com/openai/tiktoken): bộ tách token chính thức mà notebook so khớp.
- A. Petrov và cộng sự (2023), [Language Model Tokenizers Introduce Unfairness Between Languages](https://arxiv.org/abs/2305.15425).
- NLLB Team (2022), [No Language Left Behind](https://arxiv.org/abs/2207.04672): bộ câu song song FLORES-200.
- T. Mikolov và cộng sự (2013), [Efficient Estimation of Word Representations in Vector Space](https://arxiv.org/abs/1301.3781),
  và T. Mikolov, W. Yih, G. Zweig (2013), [Linguistic Regularities in Continuous Space Word Representations](https://aclanthology.org/N13-1090/).
- T. Linzen (2016), [Issues in evaluating semantic spaces using word analogies](https://arxiv.org/abs/1606.07736):
  mốc “chỉ lấy c”.
- M. Nissim, R. van Noord, R. van der Goot (2020), [Fair is Better than Sensational: Man is to Doctor as Woman is to Doctor](https://arxiv.org/abs/1905.09866).
- K. Ethayarajh (2019), [How Contextual are Contextualized Word Representations?](https://arxiv.org/abs/1909.00512):
  dị hướng trong embedding của GPT-2.

**Bài tiếp theo:** [Bài 15 · RNN & LSTM](https://d3qr1cjln6qqtk.cloudfront.net/bai-hoc/rnn/): đọc dãy vectơ theo thứ
tự, mang theo một trạng thái.

In [ ]:
import platform

import matplotlib

print(f"Python {platform.python_version()} · numpy {np.__version__} · matplotlib {matplotlib.__version__}")